# Módulo 4: Bases de datos - Ejercicio de evaluación

## 1. Creamos el comando para crear la base de datos y sus tablas en el archivo SQL:

In [5]:
import mysql.connector as con
import pandas as pd
import random
from datetime import datetime, timedelta

def create_database():
    try:
        connection = con.connect(
            host="localhost",
            port="3306",
            user="root",
            password="admin"
        )
        cursor = connection.cursor()
        cursor.execute("SHOW DATABASES;")
        databases = [db[0] for db in cursor.fetchall()]
        if "supermercado" in databases:
            print("La base de datos ya existe.")
        else:
            cursor.execute("CREATE DATABASE supermercado DEFAULT CHARACTER SET utf8;")
            print("Base de datos creada exitosamente.")
        cursor.close()
        connection.close()
    except con.Error as err:
        print(f"Error: {err}")

def execute_sql():
    try:
        connection = con.connect(
            host="localhost",
            port="3306",
            user="root",
            password="admin",
            database="supermercado"
        )
        cursor = connection.cursor()
        
        tables = {
            "tiendas": "CREATE TABLE tiendas (id_tienda INT AUTO_INCREMENT PRIMARY KEY, nombre_tienda VARCHAR(100) NOT NULL, direccion VARCHAR(255) NOT NULL, ciudad VARCHAR(100) NOT NULL)",
            "empleados": "CREATE TABLE empleados (id_empleado INT AUTO_INCREMENT PRIMARY KEY, nombre_empleado VARCHAR(100) NOT NULL, puesto ENUM('Cajero', 'Gerente', 'Reponedor') NOT NULL, id_tienda INT, FOREIGN KEY (id_tienda) REFERENCES tiendas(id_tienda) ON DELETE SET NULL)",
            "categorias": "CREATE TABLE categorias (id_categoria INT AUTO_INCREMENT PRIMARY KEY, nombre_categoria VARCHAR(100) NOT NULL)",
            "productos": "CREATE TABLE productos (id_producto INT AUTO_INCREMENT PRIMARY KEY, nombre_producto VARCHAR(100) NOT NULL, precio DECIMAL(10, 2) NOT NULL, stock INT NOT NULL, id_categoria INT, FOREIGN KEY (id_categoria) REFERENCES categorias(id_categoria) ON DELETE SET NULL)",
            "clientes": "CREATE TABLE clientes (id_cliente INT AUTO_INCREMENT PRIMARY KEY, first_name VARCHAR(100) NOT NULL, last_name VARCHAR(100) NOT NULL, email VARCHAR(150) UNIQUE NOT NULL, codigo_postal VARCHAR(10) NOT NULL)",
            "ordenes": "CREATE TABLE ordenes (id_orden INT AUTO_INCREMENT PRIMARY KEY, id_cliente INT NOT NULL, id_empleado INT NOT NULL, fecha_orden DATETIME DEFAULT CURRENT_TIMESTAMP, metodo_pago ENUM('Tarjeta', 'Efectivo') NOT NULL, FOREIGN KEY (id_cliente) REFERENCES clientes(id_cliente) ON DELETE CASCADE, FOREIGN KEY (id_empleado) REFERENCES empleados(id_empleado) ON DELETE CASCADE)",
            "detalle_orden": "CREATE TABLE detalle_orden (id_detalle INT AUTO_INCREMENT PRIMARY KEY, id_orden INT NOT NULL, id_producto INT NOT NULL, cantidad INT NOT NULL, precio_unitario DECIMAL(10, 2) NOT NULL, descuento DECIMAL(5, 2) DEFAULT 0.00, FOREIGN KEY (id_orden) REFERENCES ordenes(id_orden) ON DELETE CASCADE, FOREIGN KEY (id_producto) REFERENCES productos(id_producto) ON DELETE CASCADE)"
        }
        
        cursor.execute("SHOW TABLES;")
        existing_tables = [table[0] for table in cursor.fetchall()]
        
        for table_name, create_stmt in tables.items():
            if table_name in existing_tables:
                print(f"La tabla '{table_name}' ya existe.")
            else:
                cursor.execute(create_stmt)
                print(f"Tabla '{table_name}' creada exitosamente.")
        
        cursor.close()
        connection.close()
    except con.Error as err:
        print(f"Error: {err}")

if __name__ == "__main__":
    create_database()
    execute_sql()
    print("Proceso finalizado.")

Base de datos creada exitosamente.
Tabla 'tiendas' creada exitosamente.
Tabla 'empleados' creada exitosamente.
Tabla 'categorias' creada exitosamente.
Tabla 'productos' creada exitosamente.
Tabla 'clientes' creada exitosamente.
Tabla 'ordenes' creada exitosamente.
Tabla 'detalle_orden' creada exitosamente.
Proceso finalizado.


 ## 2: Generar datos demo desde Python

In [6]:
def get_connection():
    return con.connect(
        host="localhost",
        port="3306",
        user="root",
        password="admin",
        database="supermercado"
    )

def insert_data(table, columns, values, batch_size=1000):
    connection = get_connection()
    cursor = connection.cursor()

    try:
        placeholders = ", ".join(["%s"] * len(columns))
        column_names = ", ".join(columns)
        
        query = f"""
            REPLACE INTO {table} ({column_names}) 
            VALUES ({placeholders});
        """

        # Dividir los datos en lotes
        for i in range(0, len(values), batch_size):
            batch = values[i : i + batch_size]
            cursor.executemany(query, batch)
            connection.commit()  # Confirmar después de cada lote

        print(f"Datos insertados en {table} correctamente.")
    
    except Exception as e:
        connection.rollback()
        print(f"Error en {table}: {e}")
    
    finally:
        cursor.close()
        connection.close()

def generate_tiendas():
    tiendas = [(i, f"Super {nombre}", f"Calle {i} Centro", random.choice(["Madrid", "Barcelona", "México DF"])) 
               for i, nombre in enumerate(["Norte", "Sur", "Este", "Oeste", "Centro"], 1)]
    insert_data("tiendas", ["id_tienda", "nombre_tienda", "direccion", "ciudad"], tiendas)

def generate_empleados():
    puestos = ['Cajero', 'Gerente', 'Reponedor']
    empleados = [(i, f"Empleado {i}", random.choice(puestos), random.randint(1, 5)) for i in range(1, 101)]
    insert_data("empleados", ["id_empleado", "nombre_empleado", "puesto", "id_tienda"], empleados)

def generate_categorias():
    categorias = [(i, nombre) for i, nombre in enumerate(["Lácteos", "Carnes", "Frutas", "Verduras", "Bebidas", "Snacks"], 1)]
    insert_data("categorias", ["id_categoria", "nombre_categoria"], categorias)

def generate_productos():
    productos = [(i, f"Producto {i}", round(random.uniform(0.5, 50.0), 2), random.randint(0, 500), random.randint(1, 6)) for i in range(1, 25)]
    insert_data("productos", ["id_producto", "nombre_producto", "precio", "stock", "id_categoria"], productos)

def generate_clientes():
    clientes = [(i, f"Cliente {i}", f"Apellido {i}", f"cliente{i}@mail.com", f"{random.randint(10000, 99999)}") for i in range(1, 2001)]
    insert_data("clientes", ["id_cliente", "first_name", "last_name", "email", "codigo_postal"], clientes)

def generate_ordenes():
    ordenes = [(i, random.randint(1, 2000), random.randint(1, 100), datetime.now() - timedelta(days=random.randint(1, 365)), random.choice(['Tarjeta', 'Efectivo'])) for i in range(1, 10001)]
    insert_data("ordenes", ["id_orden", "id_cliente", "id_empleado", "fecha_orden", "metodo_pago"], ordenes)

def generate_detalle_orden():
    detalles = [(i, random.randint(1, 10000), random.randint(1, 24), random.randint(1, 20), round(random.uniform(0.5, 50.0), 2), round(random.uniform(0, 5), 2)) for i in range(1, 30001)]
    insert_data("detalle_orden", ["id_detalle", "id_orden", "id_producto", "cantidad", "precio_unitario", "descuento"], detalles)

if __name__ == "__main__":
    generate_tiendas()
    generate_empleados()
    generate_categorias()
    generate_productos()
    generate_clientes()
    generate_ordenes()
    generate_detalle_orden()
    print("Todos los datos han sido generados exitosamente.")

Datos insertados en tiendas correctamente.
Datos insertados en empleados correctamente.
Datos insertados en categorias correctamente.
Datos insertados en productos correctamente.
Datos insertados en clientes correctamente.
Datos insertados en ordenes correctamente.
Datos insertados en detalle_orden correctamente.
Todos los datos han sido generados exitosamente.


## 3: Consultas SQL

### 1 - Listado de órdenes con detalles de cliente y empleado:

```sql
SELECT 
    o.id_orden, 
    o.fecha_orden, 
    c.first_name AS cliente_nombre, 
    c.last_name AS cliente_apellido, 
    e.nombre_empleado, 
    o.metodo_pago
FROM 
    ordenes o
JOIN 
    clientes c ON o.id_cliente = c.id_cliente
JOIN 
    empleados e ON o.id_empleado = e.id_empleado;

```
### 2 - Productos con stock bajo (menor a 10):

```sql
SELECT 
    p.nombre_producto, 
    ca.nombre_categoria, 
    p.stock
FROM 
    productos p
JOIN 
    categorias ca ON p.id_categoria = ca.id_categoria
WHERE 
    p.stock < 10;
```

### 3 - Ventas totales por categoría:

```sql
SELECT 
    ca.nombre_categoria, 
    ROUND(SUM(do.cantidad * do.precio_unitario)) AS ventas_totales
FROM 
    detalle_orden do
JOIN 
    productos p ON do.id_producto = p.id_producto
JOIN 
    categorias ca ON p.id_categoria = ca.id_categoria
GROUP BY 
    ca.id_categoria;
```

### 4 - Clientes con mayores gastos acumulados:

```sql
SELECT 
    c.first_name AS cliente_nombre, 
    c.last_name AS cliente_apellido, 
    SUM(do.cantidad * do.precio_unitario - IFNULL(do.descuento, 0)) AS gasto_total
FROM 
    detalle_orden do
JOIN 
    ordenes o ON do.id_orden = o.id_orden
JOIN 
    clientes c ON o.id_cliente = c.id_cliente
GROUP BY 
    c.id_cliente
ORDER BY 
    gasto_total DESC;

```

### 5 - Empleados y número de órdenes gestionadas:

```sql
SELECT 
    e.nombre_empleado, 
    e.puesto, 
    COUNT(o.id_orden) AS num_ordenes
FROM 
    empleados e
LEFT JOIN 
    ordenes o ON e.id_empleado = o.id_empleado
GROUP BY 
    e.id_empleado;
```

### 6 - Órdenes filtradas por fecha y tienda:

```sql
SELECT 
    o.id_orden, 
    o.fecha_orden, 
    t.nombre_tienda, 
    c.first_name AS cliente_nombre, 
    c.last_name AS cliente_apellido
FROM 
    ordenes o
JOIN 
    tiendas t ON o.id_empleado = t.id_tienda
JOIN 
    clientes c ON o.id_cliente = c.id_cliente
WHERE 
    o.fecha_orden BETWEEN '2025-01-01' AND '2025-01-31'
    AND t.id_tienda = 1;
```

### 7 - Ranking de productos más vendidos en cada tienda:

```sql
SELECT 
    t.nombre_tienda, 
    p.nombre_producto, 
    SUM(do.cantidad) AS total_vendido
FROM 
    detalle_orden do
JOIN 
    ordenes o ON do.id_orden = o.id_orden
JOIN 
    empleados e ON o.id_empleado = e.id_empleado
JOIN 
    tiendas t ON e.id_tienda = t.id_tienda
JOIN 
    productos p ON do.id_producto = p.id_producto
GROUP BY 
    t.id_tienda, p.id_producto
ORDER BY 
    t.id_tienda, total_vendido DESC
LIMIT 3;
```

### 8 - Total de ventas por cliente considerando los productos vendidos:

```sql
SELECT 
    c.first_name AS cliente_nombre, 
    c.last_name AS cliente_apellido, 
    FLOOR(
        (SELECT SUM(do.cantidad * do.precio_unitario - IFNULL(do.descuento, 0))
         FROM detalle_orden do
         JOIN ordenes o ON do.id_orden = o.id_orden
         WHERE o.id_cliente = c.id_cliente)
    ) AS total_ventas
FROM 
    clientes c
ORDER BY 
    total_ventas DESC;
```

cursor.execute('''
select * from customers;
''')